In [ ]:
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, get_scheduler
import peft
from peft import LoraConfig, get_peft_model
from datasets import DatasetDict, Dataset
from transformers import AdamW, get_linear_schedule_with_warmup
import torch.nn as nn
import numpy as np
import pandas as pd
import csv
from tqdm.auto import tqdm

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
train_df= pd.read_csv('/content/drive/MyDrive/w266-Project/TextGenerate Summary/ESM_Train_filteredDescription.csv')
eval_df = pd.read_csv('/content/drive/MyDrive/w266-Project/TextGenerate Summary/ESM_eval_filteredDescription.csv')

train_df = train_df.drop(columns=['key'], errors='ignore')
val_df = eval_df.drop(columns=['key'], errors='ignore')

train_df = train_df.rename(columns={
    'filtered_description': 'source',
    'concatenated_description': 'summary_text'
})

val_df = val_df.rename(columns={
    'filtered_description': 'source',
    'concatenated_description': 'summary_text'
})


train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset
})



In [ ]:
def tokenize_function(examples):
    source_encoding = tokenizer(examples['source'], padding='max_length', truncation=True, max_length=200)
    summary_encoding = tokenizer(examples['summary_text'], padding='max_length', truncation=True, max_length=200)
    return {
        'input_ids': source_encoding['input_ids'],
        'attention_mask': source_encoding['attention_mask'],
        'labels': summary_encoding['input_ids']
    }

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)


Map:   0%|          | 0/48251 [00:00<?, ? examples/s]

Map:   0%|          | 0/3513 [00:00<?, ? examples/s]

In [ ]:
'''
for batch in train_loader:
    print({k: v.shape for k, v in batch.items()})
    break

'''

{'input_ids': torch.Size([20, 512]), 'attention_mask': torch.Size([20, 512]), 'labels': torch.Size([20, 512])}


In [ ]:
'''
def collate_fn(batch):
    input_ids = [torch.tensor(item['input_ids']) for item in batch]
    attention_mask = [torch.tensor(item['attention_mask']) for item in batch]
    labels = [torch.tensor(item['labels']) for item in batch]

    input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence(attention_mask, batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)  # -100 to ignore tokens

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }
  '''

In [ ]:
'''
batch_size = 20
train_loader = DataLoader(tokenized_datasets['train'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(tokenized_datasets['validation'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
'''

In [ ]:
'''
def tokenize_function(texts, summaries):
    encodings = tokenizer(texts, padding="max_length", truncation=True, max_length=512, return_tensors='pt')
    target_encodings = tokenizer(summaries, padding="max_length", truncation=True, max_length=512, return_tensors='pt')
    encodings['labels'] = target_encodings['input_ids']
    return encodings

train_encodings = tokenize_function(train_texts, train_summaries)
val_encodings = tokenize_function()
'''

In [ ]:
for param in model.parameters():
    param.requires_grad = False


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
peft_config = LoraConfig(
    r=4,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['c_attn', 'c_proj'],
    task_type=TaskType.CAUSAL_LM
)
peft_model = get_peft_model(model, peft_config)

/usr/local/lib/python3.10/dist-packages/peft/tuners/lora/layer.py:1119: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
peft_model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2SdpaAttention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D()
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
      

In [ ]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params

total_params, trainable_params = count_parameters(model)

print(total_params)
print(trainable_params)

124845312
405504


In [ ]:
'''
optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 3
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    "cosine",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps
)
'''

In [ ]:
'''
for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Training")
    for batch in train_loader:
        inputs = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        loss.backward()


        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        train_pbar.set_postfix({'loss': loss.item()})

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation")
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)
            loss = outputs.loss

            val_loss += loss.item()
            val_pbar.set_postfix({'loss': loss.item()})

    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
  '''

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy="epoch",
    learning_rate=5e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir='./logs',
    logging_steps=10,
    fp16=True,
    report_to= None,

)

In [ ]:
from transformers import EvalPrediction
import numpy as np

def compute_metrics(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    # Add your metric computation here, for example using ROUGE
    # For cross-entropy, we can use the loss directly
    return {"loss": np.mean(logits)}

In [ ]:
'''
tokenizer.pad_token = tokenizer.eos_token
'''

In [ ]:
#data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
)

In [ ]:
import gc
def clear_gpu_memory():
    torch.cuda.empty_cache()
    gc.collect()

clear_gpu_memory()

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss
1,4.287800,4.324479
2,4.432900,4.246082
3,4.175500,4.170001


{'eval_loss': 4.1700005531311035,
 'eval_runtime': 38.9157,
 'eval_samples_per_second': 90.272,
 'eval_steps_per_second': 45.149,
 'epoch': 3.0}

In [ ]:
from huggingface_hub import notebook_login

In [ ]:
notebook_login()

In [ ]:
trainer.push_to_hub("ESM2-cosineSim-GPT2-LoRA")

adapter_model.safetensors:   0%|          | 0.00/1.63M [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

training_args.bin:   0%|          | 0.00/5.05k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Drrnl/results/commit/ed452ee7ab287f2705ee923df7e65806bed48c11', commit_message='ESM2-cosineSim-GPT2-LoRA', commit_description='', oid='ed452ee7ab287f2705ee923df7e65806bed48c11', pr_url=None, pr_revision=None, pr_num=None)

In [ ]:
from rake_nltk import Rake
import nltk
nltk.download('stopwords')
nltk.download('punkt')
rake_nltk_var = Rake()
text = 'Netrins control guidance of CNS commissural axons and peripheral motor axons'
rake_nltk_var.extract_keywords_from_text(text)
keyword_extracted = rake_nltk_var.get_ranked_phrases()[:5]


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
print(text)

Netrins control guidance of CNS commissural axons and peripheral motor axons


In [ ]:
join_input = ' '.join(keyword_extracted)
print(join_input)

peripheral motor axons netrins control guidance cns commissural axons


In [ ]:
print(keyword_extracted)

['high level hh pathway signaling', 'wnt signaling pathway', 'hh ), possibly', 'specifically mediating depalmitoleoylation', 'peripheral motor axons']


Netrins control guidance of CNS commissural axons and peripheral motor axons . Its association with either fra or unc-5 receptors will lead to axon attraction or repulsion, respectively . While short-range repulsion requires both fra and unc-5 receptors, long-range repulsion only requires unc-5 . Carboxylesterase that acts as a key negative regulator of the Wnt signaling pathway by specifically mediating depalmitoleoylation of WNT proteins. Serine palmitoleoylation of WNT proteins is required for efficient binding to frizzled receptors . Also acts as a regulator of long-range activity of Hedgehog (hh), possibly by regulating the switch between low and high level hh pathway signaling . High-affinity transporter for adenine.


In [ ]:
import torch
from transformers import AutoTokenizer, BioGptForCausalLM
tokenizer = AutoTokenizer.from_pretrained("microsoft/biogpt")
model = BioGptForCausalLM.from_pretrained("microsoft/biogpt")

pytorch_model.bin:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

BioGptForCausalLM(
  (biogpt): BioGptModel(
    (embed_tokens): BioGptScaledWordEmbedding(42384, 1024, padding_idx=1)
    (embed_positions): BioGptLearnedPositionalEmbedding(1026, 1024)
    (layers): ModuleList(
      (0-23): 24 x BioGptDecoderLayer(
        (self_attn): BioGptAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (activation_fn): GELUActivation()
        (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (final_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      )
    )
    (layer_norm): LayerNorm((1024

In [ ]:
input_text = "peripheral motor axons"
# Tokenize the input text
input_ids = tokenizer.encode(input_text, return_tensors='pt')

# Move input to GPU if available
input_ids = input_ids.to(device)

# Generate text
output_ids = model.generate(
    input_ids,
    max_length=200,  # Maximum length of the generated text
    num_beams=5,     # Number of beams for beam search (higher is more thorough but slower)
    no_repeat_ngram_size=2, # Prevent repeating n-grams
    top_k =100,
    #pad_token_id=tokenizer.eos_token_id,
    do_sample=False,
    early_stopping=False  # Stop when the model has generated enough text
)

# Decode the generated text
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)

peripheral motor axons.


In [ ]:
peft_model_id = "Drrnl/results"

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model1.to(device)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2SdpaAttention(
          (c_attn): lora.Linear(
            (base_layer): Conv1D()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=768, out_features=4, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=4, out_features=2304, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
          )
          (c_proj): lora.Linear(
            (base_layer): Conv1D()
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.1, inplace=

In [ ]:
input_text = "catalyzes ligation participates first step glutathione biosynthesis essential gtpase binds gdp gtp rapid nucleotide exchange plays role 16s rrna processing 30s ribosomal subunit biogenesis possibly also cell cycle regulation energy metabolism part yehabcd fimbrial operon could contribute adhesion various surfaces specific environmental niches"

# Tokenize the input text
input_ids = tokenizer.encode(input_text, return_tensors='pt')

# Move input to GPU if available
input_ids = input_ids.to(device)

# Generate text
output_ids = model.generate(
    input_ids,
    max_length=200,  # Maximum length of the generated text
    num_beams=20,     # Number of beams for beam search (higher is more thorough but slower)
    no_repeat_ngram_size=5, # Prevent repeating n-grams
    top_k =50,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    early_stopping=True  # Stop when the model has generated enough text
)

# Decode the generated text
generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(generated_text)

catalyzes ligation participates first step glutathione biosynthesis essential gtpase binds gdp gtp rapid nucleotide exchange plays role 16s rrna processing 30s ribosomal subunit biogenesis possibly also cell cycle regulation energy metabolism part yehabcd fimbrial operon could contribute adhesion various surfaces specific environmental niches cell cycle cycle cycle cycle cycle energy metabolism cycle cycle cycle cycle andimbimbimbimbimb toimbimbrial, may. cycle. cycle. cycle cycle. cycle cycle cycle cycle Cycle cycle cycle cycle cycle cycles cycle cycle cycle cycle.imb cycle cycle cycle cycleimb cycle cycleimbimbimbimb cycleimbimb cycleimb cycleimbimbimb cycle cycleimb cycleimb cycle cycle cycleimbcycle cycle cycle cycle cycle cycl cycle cycle cycle cycle cycling cycle cycle cycle cyclecycle cycle cycle cycle Cycle Cycle cycle cycle cyclecycle Cycle cycle cycle cycle Cyclecycle cycle cycle cyclecycle cycles cycle cycle cyclecyclecycle cycle cycle cycle cyclescycle cycle cycle cycle) cy